# Python Code Review Practice — Find the Bug

Interview prep for a "Python Knowledge Expert" style role: the skill being tested isn't "can you write Python," it's **can you read someone else's code, spot what's actually wrong, and explain why** — clearly and correctly.

For each snippet below:
1. Read the code and the stated *intent* (what it's supposed to do).
2. Run the code cell — see what it actually does.
3. In the markdown cell below it, write out: (a) what's wrong, (b) *why* it happens (not just "this is a bug"), (c) how you'd fix it.
4. Only then check the Answer Key at the bottom.

This mirrors the real skill: identifying root cause, not just symptom — same process as the debugging framework you already use.

## Snippet 1
**Intent:** `add_item` should return a fresh list each time you call it without passing one in.

In [2]:
def add_item(item, items=None):        # items has a default value, [], used when no list is passed in
    items.append(item)                # adds item to the end of whichever list "items" currently points to
    return items                      # hands back that same list object

print(add_item("apple"))              # first call: no list passed in, so the default is used
print(add_item("banana"))             # second call: no list passed in again, so the default is used again


AttributeError: 'NoneType' object has no attribute 'append'

**Your answer:**

(what's wrong / why / fix)

## Snippet 2
**Intent:** `last_n` should return the last `n` items of a list.

In [15]:
def last_n(items, n):
    return items[len(items) - n- 1
                 :]   # len(items) counts total elements; subtracting gives a start index;
                                         # the colon means "slice from this index to the end of the list"

print(last_n([10, 20, 30, 40, 50], 2))  # expected [40, 50]


[30, 40, 50]


**Your answer:**

(what's wrong / why / fix)

## Snippet 3
**Intent:** build a list of functions, each of which returns its own index (0, 1, 2).

In [3]:
funcs = []                          # empty list that will hold 3 small functions
for i in range(3):                  # i takes on the values 0, 1, 2 in turn
    funcs.append(lambda: i)         # lambda: i defines a tiny unnamed function that, when called, returns i

print([f() for f in funcs])         # calls each stored function and collects its return value; expected [0, 1, 2]


[2, 2, 2]


**Your answer:**

(what's wrong / why / fix)

## Snippet 4
**Intent:** remove all users whose score is below 50.

In [4]:
scores = {"alice": 80, "bob": 40, "cara": 20, "dan": 95}   # a dictionary: name -> score

for name, score in scores.items():    # .items() gives (key, value) pairs one at a time as the loop runs
    if score < 50:                    # check each score against the threshold
        del scores[name]              # del removes that key from the dictionary immediately, mid-loop

print(scores)


RuntimeError: dictionary changed size during iteration

**Your answer:**

(what's wrong / why / fix)

## Snippet 5
**Intent:** confirm that adding 0.1 three times equals 0.3.

In [ ]:
total = 0.1 + 0.1 + 0.1   # adds three floating-point numbers together
print(total == 0.3)       # == checks for exact equality between total and 0.3


**Your answer:**

(what's wrong / why / fix)

## Snippet 6
**Intent:** safely convert a list of strings to integers, skipping anything that can't convert.

In [ ]:
def to_ints(values):
    result = []                    # will collect the successfully converted numbers
    for v in values:               # v takes each string in the input list, one at a time
        try:
            result.append(int(v))  # int(v) attempts to convert the string to an integer
        except:                    # bare except: catches ANY error raised inside the try block, not just conversion failures
            pass                   # silently does nothing and moves to the next item
    return result

print(to_ints(["1", "2", "oops", "4"]))


**Your answer:**

(what's wrong / why / fix — this one runs 'correctly' but has a real code-quality flaw. What is it, and why would a code reviewer flag it?)

---
# Answer Key — no peeking until you've written your own answers above

### Snippet 1 — Mutable default argument
**Bug:** `items=[]` is created **once**, at function definition time, not on every call. Every call that doesn't pass its own list shares and mutates that *same* list object.
**Root cause:** default argument values in Python are evaluated once, at def-time, not per-call.
**Fix:** `def add_item(item, items=None): if items is None: items = []`

### Snippet 2 — Off-by-one slicing
**Bug:** `len(items) - n - 1` is one index too far left, so it grabs `n+1` items instead of `n`.
**Root cause:** classic off-by-one — conflating "index of the item n-from-the-end" with "slice start point," which should just be `len(items) - n`.
**Fix:** `return items[len(items) - n:]` (or simply `items[-n:]`).

### Snippet 3 — Late-binding closures
**Bug:** all three lambdas print `2`, not `0, 1, 2`.
**Root cause:** the lambda captures the *variable* `i` by reference, not its value at creation time. By the time the lambdas run, the loop has finished and `i` is `2` for all of them.
**Fix:** default-argument trick to capture by value: `funcs.append(lambda i=i: i)`.

### Snippet 4 — Mutating a dict while iterating it
**Bug:** raises `RuntimeError: dictionary changed size during iteration` (or silently skips entries, depending on Python version/timing).
**Root cause:** you can't safely add/remove keys from a dict while a `for` loop is actively iterating over it.
**Fix:** iterate over a copy: `for name, score in list(scores.items()):` or build a new dict via a comprehension: `{k: v for k, v in scores.items() if v >= 50}`.

### Snippet 5 — Floating point comparison
**Bug:** prints `False`. `0.1 + 0.1 + 0.1` is actually `0.30000000000000004` in binary floating point.
**Root cause:** most decimal fractions can't be represented exactly in binary floating point — this isn't a Python bug, it's how IEEE 754 floats work in every mainstream language.
**Fix:** compare with a tolerance: `abs(total - 0.3) < 1e-9`, or use the `decimal` module when exactness matters (e.g. money).

### Snippet 6 — Bare `except`
**Bug:** the code "works" (drops `"oops"`, returns `[1, 2, 4]`), but a bare `except:` catches *everything* — including `KeyboardInterrupt`, `SystemExit`, and typos/logic errors elsewhere in the try block — silently.
**Root cause:** overly broad exception handling hides bugs instead of handling the *specific* failure you intended (a `ValueError` from a bad `int()` conversion).
**Fix:** `except ValueError: pass` — catch only what you expect, let everything else surface.